# FinQA: learn from the large model's worked answers (distillation)

A large model worked through 2,348 FinQA *training* questions (figures, a short explanation, the formula); only the ones it got right are kept. Here the plain Gemma 3 4B learns from them, then answers the 297 unseen test questions three times each, 8 questions at a time. About 1.5 to 2 hours.

1. **Runtime > Change runtime type > T4 GPU**, save.
2. **Runtime > Run all**. No uploads.
3. Keep the tab open. `fin_distilled_adapter.zip` downloads after training (keep it), and `fin_distilled_results.zip` at the end: send that one to Claude.

Every log line starts with the time (India) and shows done / total.

In [ ]:
%%capture
!pip install -q unsloth
!rm -rf /content/slm-opti && git clone -q --depth 1 -b v2-split https://github.com/eshwarpotturi/slm-opti /content/slm-opti
!rm -rf /content/FinQA && git clone -q --depth 1 https://github.com/czyssrs/FinQA /content/FinQA

In [ ]:
import os, sys, json, time, re, random, zipfile, collections
os.environ["TZ"] = "Asia/Kolkata"; time.tzset()
os.chdir("/content/slm-opti"); sys.path.insert(0, "/content/slm-opti"); sys.path.insert(0, "/content/slm-opti/fin")
def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)
import torch
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > T4 GPU, then Runtime > Run all."
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(model_name="unsloth/gemma-3-4b-it", max_seq_length=3072, load_in_4bit=True, full_finetuning=False)
import run, prep
log("model loaded")

## Training examples: the large model's worked answers

In [ ]:
raw = {x["id"]: x for x in json.load(open("/content/FinQA/dataset/train.json"))}
kept = [json.loads(l) for l in open("fin/data/distill.jsonl")]
kept = [r for r in kept if r["keep"] and r["id"] in raw]
def tidy(reply):
    # the calculator does the arithmetic, so the formula line stops before any '= result'
    return re.sub(r"(FORMULA\s*:[^\n=]*)=[^\n]*", lambda m: m.group(1).rstrip(), reply.strip())
from datasets import Dataset
rows = []
for r in kept:
    t = prep.slim(raw[r["id"]])
    msgs = [{"role": "user", "content": [{"type": "text", "text": run.TUNED_DIST + "\n\n" + run.ask(t)}]},
            {"role": "assistant", "content": [{"type": "text", "text": tidy(r["reply"])}]}]
    rows.append({"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")})
random.Random(7).shuffle(rows)
ds = Dataset.from_list(rows)
log(len(rows), "training examples")
print(tidy(kept[0]["reply"]))

In [ ]:
model = FastModel.get_peft_model(model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407)
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from unsloth.chat_templates import train_on_responses_only
class Stamp(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs:
            log("step %d/%d  loss %.3f" % (state.global_step, state.max_steps, logs["loss"]))
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, callbacks=[Stamp()], args=SFTConfig(
    dataset_text_field="text", per_device_train_batch_size=1, gradient_accumulation_steps=8, warmup_steps=10,
    num_train_epochs=1, learning_rate=2e-4, logging_steps=5, optim="adamw_8bit", weight_decay=0.01,
    lr_scheduler_type="linear", seed=3407, report_to="none", output_dir="outputs", disable_tqdm=True))
trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n", response_part="<start_of_turn>model\n")
os.makedirs("fin/results", exist_ok=True)
t0 = time.time(); trainer.train()
json.dump({"minutes": round((time.time() - t0) / 60, 1), "examples": len(rows), "log": trainer.state.log_history},
          open("fin/results/finetune_log_distilled.json", "w"))
log("training took %.0f minutes" % ((time.time() - t0) / 60))
model.save_pretrained("/content/fin_distilled_adapter")
import shutil
shutil.make_archive("/content/fin_distilled_adapter", "zip", "/content/fin_distilled_adapter")
from google.colab import files
files.download("/content/fin_distilled_adapter.zip")

## The 297 unseen questions: three answers each, 8 questions at a time

In [ ]:
FastModel.for_inference(model)
tok = getattr(tokenizer, "tokenizer", tokenizer)
tok.padding_side = "left"
tests = json.load(open("fin/data/test.json"))
OUT = "fin/results/gemma-3-4b-distilled__L24-dist.jsonl"
done = {json.loads(l)["id"] for l in open(OUT)} if os.path.exists(OUT) else set()
todo = sorted([t for t in tests if t["id"] not in done], key=lambda t: len(run.render(t)))
BATCH = 8

def prompt(t):
    msgs = [{"role": "user", "content": [{"type": "text", "text": run.TUNED_DIST + "\n\n" + run.ask(t)}]}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(texts, sample, n):
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    kw = dict(max_new_tokens=350, num_return_sequences=n, pad_token_id=tok.pad_token_id)
    kw.update(do_sample=True, temperature=0.7) if sample else kw.update(do_sample=False)
    out = model.generate(**enc, **kw)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

def read(text):
    yn = re.findall(r"ANSWER\s*:\s*\**\s*(yes|no)\b", text or "", flags=re.I)
    if yn:
        return yn[-1].lower(), None
    f = run.read_formula(text)
    if not f:
        return None, None
    f = f.split("=")[0].strip()
    try:
        v = run.calc(f)
        return (None if isinstance(v, str) else v), f
    except Exception:
        return None, f

def key(a):
    return "%.3g" % a if isinstance(a, float) else a

log("testing: %d to do, %d already done" % (len(todo), len(done)))
t0 = time.time()
for i in range(0, len(todo), BATCH):
    part = todo[i:i + BATCH]
    texts = [prompt(t) for t in part]
    t1 = time.time()
    firsts = generate(texts, False, 1)
    others = generate(texts, True, 2)
    for j, t in enumerate(part):
        replies = [firsts[j], others[2 * j], others[2 * j + 1]]
        tries = [read(r) for r in replies]
        counts = collections.Counter(key(a) for a, _ in tries if a is not None)
        if counts:
            top = counts.most_common(1)[0][0]
            a, f = next(x for x in tries if x[0] is not None and key(x[0]) == top)
        else:
            a, f = None, None
        rec = {"id": t["id"], "gold": t["answer"], "answer": a, "formula": f, "right": run.correct(a, t["answer"]),
               "first": tries[0][0], "first_formula": tries[0][1], "first_right": run.correct(tries[0][0], t["answer"]),
               "agree": counts[top] if counts else 0, "votes": list(counts.items()), "raw": replies[0][-800:], "cost": 0.0,
               "secs": round((time.time() - t1) / len(part), 1)}
        open(OUT, "a").write(json.dumps(rec) + "\n")
    rs = [json.loads(l) for l in open(OUT)]
    log("%d/%d done  first answer right %d (%.0f%%)  batch %.0fs" % (len(rs), len(tests), sum(r["first_right"] for r in rs),
        100 * sum(r["first_right"] for r in rs) / len(rs), time.time() - t1))
log("testing took %.0f minutes" % ((time.time() - t0) / 60))

In [ ]:
rs = [json.loads(l) for l in open(OUT)]
first = sum(r["first_right"] for r in rs)
sure = [r for r in rs if r["agree"] == 3]
log("first answer correct: %d of %d (%.0f%%)" % (first, len(rs), 100 * first / len(rs)))
log("all three answers agree on %d questions; %d of those are correct" % (len(sure), sum(r["right"] for r in sure)))
with zipfile.ZipFile("/content/fin_distilled_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in [OUT, "fin/results/finetune_log_distilled.json"]:
        z.write(f)
from google.colab import files
files.download("/content/fin_distilled_results.zip")